In [0]:
from pyspark.sql.functions import sum, countDistinct, avg, col,lit
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import datediff, when
from pyspark.sql.functions import date_format

In [0]:
%python
customers_df = spark.read.format("delta").load("/mnt/data/silver/customers")
orders_df = spark.read.format("delta").load("/mnt/data/silver/orders")
products_df = spark.read.format("delta").load("/mnt/data/silver/products")

In [0]:
revenue_by_status = (orders_df.groupBy("orderStatus")
                                    .agg(
                                            F.sum("lineTotal").alias("totalRevenue"),
                                            F.avg("lineTotal").alias("avgOrderValue"),
                                            F.sum("lineMargin").alias("totalMargin"),
                                            F.countDistinct("orderId").alias("orderCount")
                                        )
                                    .withColumn("marginPct", (F.col("totalMargin") / F.col("totalRevenue")) * 100)
                                    .orderBy(F.col("totalRevenue").desc())
                                )

display(revenue_by_status)

orderStatus,totalRevenue,avgOrderValue,totalMargin,orderCount,marginPct
SHIPPED,13125.223,1312.5223,3781.973,9,28.814542808148857
PROCESSING,2247.0,2247.0,882.0,1,39.25233644859813
PENDING,1800.0,1800.0,434.5,1,24.138888888888886


In [0]:
# Aggregate revenue, margin, and counts by customer segment
segment_agg = (orders_df.groupBy("customerSegment")\
                .agg(
                      F.sum("lineTotal").alias("total_revenue"),
                      F.sum("lineMargin").alias("total_margin"),
                      F.countDistinct("customerSK").alias("customer_count"))
)

display(segment_agg)

# Average revenue per customer
segment_agg_1 = segment_agg.withColumn("avg_revenue_per_customer",
                        F.col("total_revenue") / F.col("customer_count"))

display(segment_agg_1)


# Revenue concentration: % share of each segment from the total revenue
total_rev = segment_agg.agg(F.sum("total_revenue").alias("overall_rev")).collect()[0]["overall_rev"]

segment_agg_2 = segment_agg.withColumn(
    "revenue_percentage",
    (F.col("total_revenue") / F.lit(total_rev)) * 100
)

display(segment_agg_2)

# Top 3 segments by margin using window
w = Window.orderBy(F.col("total_margin").desc())

segment_agg = segment_agg.withColumn("rank_margin", F.rank().over(w))

top_3_segments = segment_agg.filter(F.col("rank_margin") <= 3)
display(top_3_segments)

# Final ordered results
final_result = segment_agg.orderBy(F.col("total_revenue").desc())

display(final_result)
print("\nTop 3 customer segments by margin:")
display(top_3_segments)

customerSegment,total_revenue,total_margin,customer_count
NORTH AMERICA_MID-MARKET,3828.1530000000002,1072.1529999999998,2
NORTH AMERICA_STRATEGIC,558.005,103.005,1
LATIN AMERICA_SMB,2247.0,882.0,1
EUROPE_STRATEGIC,1257.21,415.71000000000004,1
NORTH AMERICA_ENTERPRISE,7164.37,2078.62,2
EUROPE_MID-MARKET,2117.485,546.985,1


customerSegment,total_revenue,total_margin,customer_count,avg_revenue_per_customer
NORTH AMERICA_MID-MARKET,3828.1530000000002,1072.1529999999998,2,1914.0765000000001
NORTH AMERICA_STRATEGIC,558.005,103.005,1,558.005
LATIN AMERICA_SMB,2247.0,882.0,1,2247.0
EUROPE_STRATEGIC,1257.21,415.71000000000004,1,1257.21
NORTH AMERICA_ENTERPRISE,7164.37,2078.62,2,3582.185
EUROPE_MID-MARKET,2117.485,546.985,1,2117.485


customerSegment,total_revenue,total_margin,customer_count,revenue_percentage
NORTH AMERICA_MID-MARKET,3828.1530000000002,1072.1529999999998,2,22.292704910715404
NORTH AMERICA_STRATEGIC,558.005,103.005,1,3.2494628097946316
LATIN AMERICA_SMB,2247.0,882.0,1,13.085085140112609
EUROPE_STRATEGIC,1257.21,415.71000000000004,1,7.321183751224288
NORTH AMERICA_ENTERPRISE,7164.37,2078.62,2,41.72069044293217
EUROPE_MID-MARKET,2117.485,546.985,1,12.330872945220895


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


customerSegment,total_revenue,total_margin,customer_count,rank_margin
NORTH AMERICA_ENTERPRISE,7164.37,2078.62,2,1
NORTH AMERICA_MID-MARKET,3828.1530000000002,1072.1529999999998,2,2
LATIN AMERICA_SMB,2247.0,882.0,1,3


customerSegment,total_revenue,total_margin,customer_count,rank_margin
NORTH AMERICA_ENTERPRISE,7164.37,2078.62,2,1
NORTH AMERICA_MID-MARKET,3828.1530000000002,1072.1529999999998,2,2
LATIN AMERICA_SMB,2247.0,882.0,1,3
EUROPE_MID-MARKET,2117.485,546.985,1,4
EUROPE_STRATEGIC,1257.21,415.71000000000004,1,5
NORTH AMERICA_STRATEGIC,558.005,103.005,1,6



Top 3 customer segments by margin:


customerSegment,total_revenue,total_margin,customer_count,rank_margin
NORTH AMERICA_ENTERPRISE,7164.37,2078.62,2,1
NORTH AMERICA_MID-MARKET,3828.1530000000002,1072.1529999999998,2,2
LATIN AMERICA_SMB,2247.0,882.0,1,3


In [0]:
##Overall Metrics KPI

overall_metrics = orders_df.agg(
                                sum("lineTotal").alias("total_revenue"),
                                countDistinct("orderId").alias("total_orders"),
                                (sum("lineTotal") / countDistinct("orderId")).alias("avg_order_value"),
                                (sum("lineMargin") / sum("lineTotal") * 100).alias("overall_margin_pct")
)

display(overall_metrics)

total_revenue,total_orders,avg_order_value,overall_margin_pct
17172.222999999998,11,1561.1111818181816,29.690232883651696


In [0]:
##Top 5 customer KPI

top_customers = orders_df.groupBy("customerSK")\
    .agg(sum("lineTotal").alias("total_revenue"))\
    .join(customers_df, "customerSK", "left")\
    .orderBy(col("total_revenue").desc())\
    .limit(5)

display(top_customers)

customerSK,total_revenue,customerId,customerName,industry,region,customerTier,isActive,email,phone,address,city,state,postalCode,onboardDate,creditRating,fullAddress,customerSegment,customerAge,processedAt
1,6214.42,101,ACME CORP,MANUFACTURING,NORTH AMERICA,ENTERPRISE,true,JOHN.DOE@ACME.COM,+1-555-0101,123 MAIN ST,NEW YORK,NY,10001,2020-01-15,A,123 MAIN ST | NEW YORK | NY | 10001,NORTH AMERICA_ENTERPRISE,2115,2025-10-30T20:15:18.251Z
5,2989.203,105,FRONTIER EDUCATION,EDUCATION,NORTH AMERICA,MID-MARKET,true,HELLO@FRONTIER.EDU,555-0105,2020 CAMPUS DR,AUSTIN,TX,73301,2022-02-28,A,2020 CAMPUS DR | AUSTIN | TX | 73301,NORTH AMERICA_MID-MARKET,1340,2025-10-30T20:15:18.251Z
6,2247.0,106,GAMMA STORES,RETAIL,LATIN AMERICA,SMB,true,VENTAS@GAMMA.COM,+52-555-0106,AV. PRINCIPAL 100,MEXICO CITY,DF,01000,2020-09-12,B,AV. PRINCIPAL 100 | MEXICO CITY | DF | 01000,LATIN AMERICA_SMB,1874,2025-10-30T20:15:18.251Z
8,2117.485,108,INNOVA LABS,TECHNOLOGY,EUROPE,MID-MARKET,true,INFO@INNOVA.DE,+49-30-5550108,TECH STR. 42,BERLIN,BE,10115,2021-12-03,A,TECH STR. 42 | BERLIN | BE | 10115,EUROPE_MID-MARKET,1427,2025-10-30T20:15:18.251Z
7,1257.21,107,HELIOS ENERGY,ENERGY,EUROPE,STRATEGIC,true,CONTACT@HELIOS.EU,+44-20-5550107,10 ENERGY PLAZA,LONDON,,SW1A 1AA,2019-06-18,A,10 ENERGY PLAZA | LONDON | | SW1A 1AA,EUROPE_STRATEGIC,2326,2025-10-30T20:15:18.251Z


In [0]:
#Top 5 product KPI

top_products = (
    orders_df.groupBy("productId")
    .agg(sum("lineTotal").alias("total_revenue"))
    .join(products_df, "productId", "left")
    .orderBy(col("total_revenue").desc())
    .limit(5)
)

display(top_products)

productId,total_revenue,productName,category,unitCost,listPrice,status,launchDate,productCode,screenSize,processor,productFamily,grossMarginPct,productSK,processedAt
1001,6936.66,PHOTON LAPTOP 14,COMPUTING,865.5,1299.0,ACTIVE,2023-01-15,LT,15.6,INTEL I7,COMPUTING_LT,33.37182448036952,1,2025-10-31T05:00:58.716Z
1004,2805.005,AURORA TABLET 11,COMPUTING,455.0,749.0,ACTIVE,2023-04-05,TB,11.0,ARM PROCESSOR,COMPUTING_TB,39.25233644859813,4,2025-10-31T05:00:58.716Z
1005,2196.11,SOLARDOCK PRO,ACCESSORIES,140.25,229.0,DISCONTINUED,2022-12-01,DK,0,USB-C HUB,ACCESSORIES_DK,38.75545851528384,5,2025-10-31T05:00:58.716Z
1006,1800.0,ION SERVER BLADE,COMPUTING,682.75,1125.0,ACTIVE,2023-05-18,SV,1U,XEON GOLD,COMPUTING_SV,39.31111111111111,6,2025-10-31T05:00:58.716Z
1002,1781.5350000000003,LUMEN MONITOR 27,ACCESSORIES,205.0,329.0,ACTIVE,2023-02-20,MN,27.0,4K DISPLAY,ACCESSORIES_MN,37.68996960486322,2,2025-10-31T05:00:58.716Z


In [0]:
##Month on month sales KPI

sales_monthly = (
    orders_df
    .withColumn("month", date_format("orderDate", "yyyy-MM"))
    .groupBy("month")
    .agg(
        sum("lineTotal").alias("total_revenue"),
        countDistinct("orderId").alias("order_count"),
        (sum("lineTotal") / countDistinct("orderId")).alias("avg_order_value"),
    )
    .orderBy("month")
)

display(sales_monthly)

month,total_revenue,order_count,avg_order_value
2024-01,15915.838000000002,9,1768.4264444444445
2024-02,1256.385,2,628.1925


In [0]:
##Shipping performance KpI

shipping_df = orders_df.withColumn("days_to_ship",\
                        when(col("shipDate")\
                       .isNotNull(), datediff("shipDate", "orderDate")))

shipping_performance = shipping_df.agg(avg("days_to_ship")\
                                  .alias("avg_days_to_ship"),
                                   (sum(when(col("orderStatus") == "Shipped", 1)
                                  .otherwise(0)) / countDistinct("orderId") * 100)
                                  .alias("pct_orders_shipped"),
                                   (sum(when(col("orderStatus").isin("Pending", "Processing"), 1).otherwise(0))/ countDistinct("orderId") * 100).alias("pct_orders_pending_processing"))

display(shipping_performance)

avg_days_to_ship,pct_orders_shipped,pct_orders_pending_processing
2.6,0.0,0.0
